# Smart City AI - Long-Horizon Validation (1000 Steps)

**Objective**: Comprehensive system stability assessment with all subsystems enabled (ACO, PINN, Emergency Corridor, Predictive Green Wave, Traffic Clearing) over 1000 simulation steps.

**Date**: June 1, 2026  
**Duration**: 1000 simulation steps  
**Expected Runtime**: ~15-30 minutes


## Section 1: Import Required Libraries and Configuration


In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import traceback
import time
import warnings
import importlib.util

# Set random seeds for reproducibility
np.random.seed(42)

# Add workspace to path
workspace_root = Path("c:/Users/ddnan/intenrship_IIIT")
smart_city_path = workspace_root / "smart_city_ai"
sys.path.insert(0, str(workspace_root))
sys.path.insert(0, str(smart_city_path))

# Dynamically load the HybridEmergencyController module
try:
    spec = importlib.util.spec_from_file_location(
        "hybrid_emergency_controller",
        str(smart_city_path / "integration" / "hybrid_emergency_controller.py")
    )
    hec_module = importlib.util.module_from_spec(spec)
    sys.modules['hybrid_emergency_controller'] = hec_module
    spec.loader.exec_module(hec_module)
    HybridEmergencyController = hec_module.HybridEmergencyController
    print("✓ HybridEmergencyController loaded successfully")
except Exception as e:
    print(f"⚠️ Import error (non-critical): {e}")
    print("   Will attempt to load via fallback...")
    HybridEmergencyController = None

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")
print(f"✓ Workspace root: {workspace_root}")
print(f"✓ Smart City AI path: {smart_city_path}")
print(f"✓ Random seeds initialized for reproducibility")

✓ All libraries imported successfully
✓ Workspace root: c:\Users\ddnan\intenrship_IIIT
✓ Random seeds initialized for reproducibility


## Section 2: Initialize SUMO Simulation Environment


In [2]:
# Configuration parameters
SIMULATION_DURATION = 1000  # Steps
CHECKPOINT_INTERVAL = 100   # Steps between checkpoints
OUTPUT_DIR = workspace_root / "results" / "long_horizon_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# SUMO configuration
SUMO_CONFIG = workspace_root / "smart_city_ai" / "sumo_network" / "config.sumocfg"
PINN_MODEL_PATH = workspace_root / "smart_city_ai" / "pinn_models" / "trained_pinn.pth"

print(f"Configuration:")
print(f"  Duration: {SIMULATION_DURATION} steps")
print(f"  Checkpoint interval: {CHECKPOINT_INTERVAL} steps")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  SUMO config: {SUMO_CONFIG}")
print(f"  PINN model: {PINN_MODEL_PATH}")
print(f"  Systems: ACO, PINN, Emergency Corridor, Predictive Green Wave, Traffic Clearing")

# Verify files exist
assert SUMO_CONFIG.exists(), f"SUMO config not found: {SUMO_CONFIG}"
assert PINN_MODEL_PATH.exists(), f"PINN model not found: {PINN_MODEL_PATH}"

print("\n✓ All configuration files verified")

Configuration:
  Duration: 1000 steps
  Checkpoint interval: 100 steps
  Output directory: c:\Users\ddnan\intenrship_IIIT\results\long_horizon_validation
  SUMO config: c:\Users\ddnan\intenrship_IIIT\smart_city_ai\sumo_network\config.sumocfg
  PINN model: c:\Users\ddnan\intenrship_IIIT\smart_city_ai\pinn_models\trained_pinn.pth
  Systems: ACO, PINN, Emergency Corridor, Predictive Green Wave, Traffic Clearing

✓ All configuration files verified


In [4]:
# Initialize the HybridEmergencyController (orchestrates all systems)
print("Initializing HybridEmergencyController with all subsystems enabled...")
print("  - ACO routing: enabled")
print("  - PINN predictions: enabled")
print("  - Emergency corridor detection: enabled")
print("  - Predictive green wave: enabled")
print("  - Traffic clearing: enabled")

controller = HybridEmergencyController(
    config_file=str(SUMO_CONFIG),
    pinn_model_path=str(PINN_MODEL_PATH),
    duration=SIMULATION_DURATION,
    use_gui=False,
    pinn_interval=10,
    metrics_output_dir=str(OUTPUT_DIR),
)

print("\n✓ HybridEmergencyController initialized successfully")
print(f"  - Controller object: {type(controller).__name__}")
print(f"  - Configuration loaded with PINN model path: {PINN_MODEL_PATH.exists()}")
print(f"  - Output directory ready: {OUTPUT_DIR.exists()}")

Initializing HybridEmergencyController with all subsystems enabled...
  - ACO routing: enabled
  - PINN predictions: enabled
  - Emergency corridor detection: enabled
  - Predictive green wave: enabled
  - Traffic clearing: enabled

✓ HybridEmergencyController initialized successfully
  - Controller object: HybridEmergencyController
  - Configuration loaded with PINN model path: True
  - Output directory ready: True


## Section 3: Set Up Monitoring and Metrics Collection


In [5]:
# Initialize comprehensive metrics collection
metrics_data = {
    'step': [],
    'avg_waiting_time': [],
    'avg_queue_length': [],
    'throughput': [],
    'avg_speed': [],
    'emergency_response_time': [],
    'emergency_stop_count': [],
    'aco_reroute_count': [],
    'corridor_activations': [],
    'exception_count': [],
    'vehicles_count': [],
    'pinn_prediction_count': [],
}

# Tracking for per-checkpoint aggregation
checkpoint_metrics = []
exception_log = []
aco_reroute_events = []
emergency_events = []
corridor_activation_log = []

# Counters for run-time tracking
step_counter = 0
total_exceptions = 0
total_reroutes = 0
total_corridor_activations = 0
emergency_detection_count = 0

print("✓ Metrics collection structures initialized")
print(f"  - Tracking {len(metrics_data)} metric types")
print(f"  - Checkpoint interval: {CHECKPOINT_INTERVAL} steps")
print(f"  - Expected checkpoints: {SIMULATION_DURATION // CHECKPOINT_INTERVAL}")

✓ Metrics collection structures initialized
  - Tracking 12 metric types
  - Checkpoint interval: 100 steps
  - Expected checkpoints: 10


## Section 4: Execute 1000-Step Simulation with All Systems


In [ ]:
# Execute the 1000-step validation run
print("=" * 80)
print("STARTING 1000-STEP LONG-HORIZON VALIDATION")
print("=" * 80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

start_time = time.time()

# Run the simulation (this handles all systems internally)
try:
    print("Invoking HybridEmergencyController.run() for 1000 steps...")
    print("(All subsystems: ACO, PINN, Emergency Corridor, Predictive Green Wave, Traffic Clearing)\n")
    
    summary = controller.run()
    
    print("\n" + "=" * 80)
    print("SIMULATION COMPLETED SUCCESSFULLY")
    print("=" * 80)
    
    elapsed_time = time.time() - start_time
    print(f"Total runtime: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Extract summary metrics from controller
    if summary:
        print(f"\nSummary from controller.run():")
        for key, value in summary.items():
            if isinstance(value, (int, float)):
                print(f"  {key}: {value}")
    
except Exception as e:
    elapsed_time = time.time() - start_time
    print(f"\n❌ SIMULATION FAILED AFTER {elapsed_time:.2f} seconds")
    print(f"Error: {str(e)}")
    print(f"Traceback:\n{traceback.format_exc()}")
    exception_log.append({
        'step': 'simulation_init',
        'error': str(e),
        'traceback': traceback.format_exc()
    })
    total_exceptions += 1

STARTING 1000-STEP LONG-HORIZON VALIDATION
Start time: 2026-06-01 21:15:35

Invoking HybridEmergencyController.run() for 1000 steps...
(All subsystems: ACO, PINN, Emergency Corridor, Predictive Green Wave, Traffic Clearing)



Failed to apply route to rand_veh1: Route replacement failed for vehicle 'rand_veh1' (current edge 'hw3_to_cz2' not found in new route).
Failed to apply route to rand_veh1: Route replacement failed for vehicle 'rand_veh1' (current edge 'hw3_to_cz2' not found in new route).
Failed to apply route to rand_veh1: Route replacement failed for vehicle 'rand_veh1' (current edge 'hw3_to_cz2' not found in new route).
Failed to apply route to rand_veh1: Route replacement failed for vehicle 'rand_veh1' (current edge 'hw3_to_cz2' not found in new route).
Failed to apply route to rand_veh1: Route replacement failed for vehicle 'rand_veh1' (current edge 'hw3_to_cz2' not found in new route).
Failed to apply route to rand_veh1: Route replacement failed for vehicle 'rand_veh1' (current edge 'hw3_to_cz2' not found in new route).
Failed to apply route to rand_veh1: Route replacement failed for vehicle 'rand_veh1' (current edge 'hw3_to_cz2' not found in new route).
Failed to apply route to rand_veh1: Route

## Section 5: Aggregate and Analyze Results


In [ ]:
# Read and aggregate checkpoint metrics from simulation
print("Reading checkpoint metrics from simulation output...")

# Find checkpoint CSV files
checkpoint_files = sorted(OUTPUT_DIR.glob("*checkpoint*.csv"))
print(f"Found {len(checkpoint_files)} checkpoint files")

aggregated_metrics = {
    'avg_waiting_time': [],
    'avg_queue_length': [],
    'throughput': [],
    'avg_speed': [],
    'emergency_response_time': [],
    'emergency_stop_count': [],
    'aco_reroute_count': [],
    'corridor_activations': [],
    'pinn_prediction_count': [],
}

# Read each checkpoint file
for checkpoint_file in checkpoint_files:
    try:
        df = pd.read_csv(checkpoint_file)
        
        # Extract metrics from checkpoint
        if len(df) > 0:
            last_row = df.iloc[-1]
            
            # Collect metrics (handle missing columns gracefully)
            for metric in aggregated_metrics.keys():
                if metric in df.columns:
                    values = pd.to_numeric(df[metric], errors='coerce').dropna()
                    if len(values) > 0:
                        aggregated_metrics[metric].append(values.mean())
                    else:
                        aggregated_metrics[metric].append(0.0)
                else:
                    aggregated_metrics[metric].append(0.0)
            
            print(f"  ✓ {checkpoint_file.name}: {len(df)} rows")
    except Exception as e:
        print(f"  ✗ Error reading {checkpoint_file.name}: {e}")
        exception_log.append({
            'step': 'checkpoint_read',
            'file': str(checkpoint_file),
            'error': str(e)
        })

print(f"\n✓ Aggregated metrics from {len(checkpoint_files)} checkpoints")

In [ ]:
# Compute summary statistics
print("Computing summary statistics...")

summary_stats = {}
for metric, values in aggregated_metrics.items():
    if len(values) > 0:
        values_array = np.array(values)
        summary_stats[metric] = {
            'min': float(np.min(values_array)),
            'max': float(np.max(values_array)),
            'mean': float(np.mean(values_array)),
            'std': float(np.std(values_array)),
            'samples': len(values),
        }
    else:
        summary_stats[metric] = {
            'min': 0.0,
            'max': 0.0,
            'mean': 0.0,
            'std': 0.0,
            'samples': 0,
        }

# Print summary
print("\n" + "=" * 80)
print("AGGREGATED METRICS SUMMARY")
print("=" * 80)
for metric, stats in summary_stats.items():
    print(f"\n{metric}:")
    print(f"  Mean: {stats['mean']:.4f} | Std: {stats['std']:.4f}")
    print(f"  Min: {stats['min']:.4f} | Max: {stats['max']:.4f}")
    print(f"  Samples: {stats['samples']}")

## Section 6: Generate Validation Report


In [ ]:
# Build comprehensive validation report
report = {
    "metadata": {
        "timestamp": datetime.now().isoformat(),
        "simulation_duration_steps": SIMULATION_DURATION,
        "checkpoint_interval": CHECKPOINT_INTERVAL,
        "systems_enabled": {
            "aco": True,
            "pinn": True,
            "emergency_corridor": True,
            "predictive_green_wave": True,
            "traffic_clearing": True,
        },
        "run_time_seconds": elapsed_time if 'elapsed_time' in locals() else 0.0,
    },
    "aggregated_metrics": {},
    "per_checkpoint_timeseries": aggregated_metrics,
    "exception_log": exception_log,
    "stability_analysis": {},
}

# Add summary statistics to report
for metric, stats in summary_stats.items():
    report["aggregated_metrics"][metric] = stats

# Calculate correlation and variance metrics
print("\nCalculating stability indicators...")

# Compute coefficient of variation (CV) for each metric
stability_indicators = {}
for metric, stats in summary_stats.items():
    if stats['mean'] > 0:
        cv = stats['std'] / stats['mean']  # Coefficient of variation
    else:
        cv = 0.0
    stability_indicators[metric] = cv

report["stability_analysis"]["coefficient_of_variation"] = stability_indicators

# Exception rate
exception_rate = len(exception_log) / (SIMULATION_DURATION / CHECKPOINT_INTERVAL + 1)
report["stability_analysis"]["exception_rate"] = exception_rate
report["stability_analysis"]["total_exceptions"] = len(exception_log)

# Save report to JSON
report_path = workspace_root / "long_horizon_validation_report.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n✓ Validation report saved to: {report_path}")
print(f"  Exception rate: {exception_rate:.4f} (0.0 is best)")

## Section 7: Determine System Stability Verdict


In [ ]:
# Determine stability verdict based on comprehensive criteria
print("\n" + "=" * 80)
print("STABILITY ANALYSIS")
print("=" * 80)

# Define thresholds
THRESHOLD_EXCEPTION_RATE = 0.1  # More than 10% exceptions = unstable
THRESHOLD_CV_HIGH = 1.0         # CV > 1.0 = high variability
THRESHOLD_CV_MEDIUM = 0.5       # CV > 0.5 = medium variability
THRESHOLD_QUEUE_LENGTH_MEAN = 50  # Mean queue length > 50 = concerning
THRESHOLD_WAITING_TIME_MEAN = 60  # Mean waiting time > 60s = concerning

# Initialize verdict scoring
stability_score = 100  # Start at 100, deduct points
verdict_reasons = []

print("\nStability Criteria:")
print(f"  Exception rate threshold: {THRESHOLD_EXCEPTION_RATE * 100}%")
print(f"  High CV threshold: {THRESHOLD_CV_HIGH}")
print(f"  Queue length warning: {THRESHOLD_QUEUE_LENGTH_MEAN}")
print(f"  Waiting time warning: {THRESHOLD_WAITING_TIME_MEAN}s")

# Check exception rate
if exception_rate > THRESHOLD_EXCEPTION_RATE:
    stability_score -= 30
    verdict_reasons.append(f"High exception rate: {exception_rate:.4f}")
    print(f"\n❌ Exception rate too high: {exception_rate:.4f}")
else:
    print(f"\n✓ Exception rate acceptable: {exception_rate:.4f}")

# Check coefficient of variation for key metrics
print("\nVariability Analysis (Coefficient of Variation):")
for metric in ['avg_waiting_time', 'avg_queue_length', 'avg_speed']:
    cv = stability_indicators.get(metric, 0.0)
    print(f"  {metric}: {cv:.4f}", end="")
    
    if cv > THRESHOLD_CV_HIGH:
        print(f" ⚠️ VERY HIGH")
        stability_score -= 20
        verdict_reasons.append(f"{metric} has very high variability (CV={cv:.4f})")
    elif cv > THRESHOLD_CV_MEDIUM:
        print(f" ⚠️ MEDIUM")
        stability_score -= 10
        verdict_reasons.append(f"{metric} has medium variability (CV={cv:.4f})")
    else:
        print(f" ✓")

# Check mean values for concerning patterns
print("\nMean Value Analysis:")
avg_queue = summary_stats['avg_queue_length']['mean']
avg_wait = summary_stats['avg_waiting_time']['mean']

print(f"  Average queue length: {avg_queue:.2f}", end="")
if avg_queue > THRESHOLD_QUEUE_LENGTH_MEAN:
    print(f" ⚠️ HIGH")
    stability_score -= 15
    verdict_reasons.append(f"High average queue length: {avg_queue:.2f}")
else:
    print(f" ✓")

print(f"  Average waiting time: {avg_wait:.2f}s", end="")
if avg_wait > THRESHOLD_WAITING_TIME_MEAN:
    print(f" ⚠️ HIGH")
    stability_score -= 15
    verdict_reasons.append(f"High average waiting time: {avg_wait:.2f}s")
else:
    print(f" ✓")

# Check ACO and PINN activity
print("\nSystem Activity:")
pinn_count = summary_stats['pinn_prediction_count']['mean']
aco_reroutes = summary_stats['aco_reroute_count']['mean']

print(f"  PINN predictions (avg/checkpoint): {pinn_count:.2f}", end="")
if pinn_count > 0:
    print(f" ✓ ACTIVE")
else:
    print(f" ⚠️ INACTIVE - check integration")
    stability_score -= 10
    verdict_reasons.append("PINN predictions not recorded")

print(f"  ACO reroutes (avg/checkpoint): {aco_reroutes:.2f}", end="")
if aco_reroutes > 0:
    print(f" ✓ ACTIVE")
else:
    print(f" ⚠️ LOW - check ACO integration")
    stability_score -= 5
    verdict_reasons.append("ACO reroutes minimal")

# Determine final verdict
print("\n" + "=" * 80)
print("FINAL STABILITY VERDICT")
print("=" * 80)
print(f"Stability Score: {stability_score}/100")

if stability_score >= 70:
    verdict = "STABLE"
    emoji = "✅"
elif stability_score >= 40:
    verdict = "PARTIALLY STABLE"
    emoji = "⚠️"
else:
    verdict = "UNSTABLE"
    emoji = "❌"

print(f"\n{emoji} {verdict}")

if verdict_reasons:
    print("\nReason(s):")
    for reason in verdict_reasons:
        print(f"  - {reason}")
else:
    print("\nNo critical issues detected. System operating normally.")

# Add verdict to report
report["stability_analysis"]["verdict"] = verdict
report["stability_analysis"]["stability_score"] = stability_score
report["stability_analysis"]["reasons"] = verdict_reasons

# Save updated report
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n✓ Updated report saved: {report_path}")

## Section 8: Visualize Key Metrics and Trends


In [ ]:
# Create comprehensive visualizations
print("Generating visualization plots...")

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle('Smart City AI - 1000 Step Long-Horizon Validation', fontsize=16, fontweight='bold')

# Define checkpoint indices
checkpoint_idx = np.arange(len(aggregated_metrics['avg_waiting_time']))

# Plot 1: Average Waiting Time
ax = axes[0, 0]
ax.plot(checkpoint_idx, aggregated_metrics['avg_waiting_time'], 'b-', marker='o', markersize=4)
ax.set_title('Average Waiting Time', fontweight='bold')
ax.set_ylabel('Time (s)')
ax.set_xlabel('Checkpoint')
ax.grid(True, alpha=0.3)

# Plot 2: Average Queue Length
ax = axes[0, 1]
ax.plot(checkpoint_idx, aggregated_metrics['avg_queue_length'], 'g-', marker='s', markersize=4)
ax.set_title('Average Queue Length', fontweight='bold')
ax.set_ylabel('Queue Length')
ax.set_xlabel('Checkpoint')
ax.grid(True, alpha=0.3)

# Plot 3: Average Speed
ax = axes[0, 2]
ax.plot(checkpoint_idx, aggregated_metrics['avg_speed'], 'r-', marker='^', markersize=4)
ax.set_title('Average Vehicle Speed', fontweight='bold')
ax.set_ylabel('Speed (m/s)')
ax.set_xlabel('Checkpoint')
ax.grid(True, alpha=0.3)

# Plot 4: Throughput
ax = axes[1, 0]
ax.plot(checkpoint_idx, aggregated_metrics['throughput'], 'purple', marker='d', markersize=4)
ax.set_title('Throughput', fontweight='bold')
ax.set_ylabel('Vehicles/step')
ax.set_xlabel('Checkpoint')
ax.grid(True, alpha=0.3)

# Plot 5: Emergency Response Time
ax = axes[1, 1]
ax.plot(checkpoint_idx, aggregated_metrics['emergency_response_time'], 'orange', marker='h', markersize=4)
ax.set_title('Emergency Response Time', fontweight='bold')
ax.set_ylabel('Time (s)')
ax.set_xlabel('Checkpoint')
ax.grid(True, alpha=0.3)

# Plot 6: ACO Reroute Count
ax = axes[1, 2]
ax.bar(checkpoint_idx, aggregated_metrics['aco_reroute_count'], color='teal', alpha=0.7, edgecolor='black')
ax.set_title('ACO Reroute Events per Checkpoint', fontweight='bold')
ax.set_ylabel('Reroute Count')
ax.set_xlabel('Checkpoint')
ax.grid(True, alpha=0.3, axis='y')

# Plot 7: Corridor Activations
ax = axes[2, 0]
ax.bar(checkpoint_idx, aggregated_metrics['corridor_activations'], color='navy', alpha=0.7, edgecolor='black')
ax.set_title('Emergency Corridor Activations', fontweight='bold')
ax.set_ylabel('Activation Count')
ax.set_xlabel('Checkpoint')
ax.grid(True, alpha=0.3, axis='y')

# Plot 8: PINN Prediction Count
ax = axes[2, 1]
ax.plot(checkpoint_idx, aggregated_metrics['pinn_prediction_count'], 'brown', marker='*', markersize=10)
ax.set_title('PINN Prediction Activity', fontweight='bold')
ax.set_ylabel('Prediction Count')
ax.set_xlabel('Checkpoint')
ax.grid(True, alpha=0.3)

# Plot 9: Emergency Stop Count
ax = axes[2, 2]
ax.plot(checkpoint_idx, aggregated_metrics['emergency_stop_count'], 'crimson', marker='X', markersize=6)
ax.set_title('Emergency Stop Events', fontweight='bold')
ax.set_ylabel('Stop Count')
ax.set_xlabel('Checkpoint')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = workspace_root / "long_horizon_validation_metrics.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"✓ Metrics plot saved: {plot_path}")
plt.show()

print("\n✓ Visualization complete")

## Final Summary and Conclusions


In [ ]:
print("\n" + "=" * 80)
print("LONG-HORIZON VALIDATION SUMMARY")
print("=" * 80)

print(f"\n📊 METRICS OVERVIEW:")
print(f"  ├─ Simulation Duration: {SIMULATION_DURATION} steps")
print(f"  ├─ Checkpoints: {len(checkpoint_idx)}")
print(f"  ├─ Total Runtime: {elapsed_time if 'elapsed_time' in locals() else 'N/A':.2f}s")
print(f"  └─ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n🚗 TRAFFIC METRICS:")
print(f"  ├─ Avg Waiting Time: {summary_stats['avg_waiting_time']['mean']:.2f}s ±{summary_stats['avg_waiting_time']['std']:.2f}s")
print(f"  ├─ Avg Queue Length: {summary_stats['avg_queue_length']['mean']:.2f} ±{summary_stats['avg_queue_length']['std']:.2f}")
print(f"  ├─ Throughput (avg): {summary_stats['throughput']['mean']:.2f} vehicles/step")
print(f"  └─ Avg Speed: {summary_stats['avg_speed']['mean']:.2f} m/s ±{summary_stats['avg_speed']['std']:.2f} m/s")

print(f"\n🚨 EMERGENCY SYSTEM:")
print(f"  ├─ Avg Response Time: {summary_stats['emergency_response_time']['mean']:.2f}s")
print(f"  ├─ Total Emergency Stops: {int(np.sum(aggregated_metrics['emergency_stop_count']))}")
print(f"  └─ Corridor Activations: {int(np.sum(aggregated_metrics['corridor_activations']))}")

print(f"\n🐜 ACO & PINN:")
print(f"  ├─ Total ACO Reroutes: {int(np.sum(aggregated_metrics['aco_reroute_count']))}")
print(f"  ├─ PINN Predictions (total): {int(np.sum(aggregated_metrics['pinn_prediction_count']))}")
print(f"  ├─ Avg PINN activity/checkpoint: {summary_stats['pinn_prediction_count']['mean']:.2f}")
print(f"  └─ ACO active: {'✓' if summary_stats['aco_reroute_count']['mean'] > 0 else '✗'}")

print(f"\n⚙️ SYSTEM HEALTH:")
print(f"  ├─ Total Exceptions: {len(exception_log)}")
print(f"  ├─ Exception Rate: {exception_rate:.4f} ({exception_rate*100:.2f}%)")
print(f"  ├─ Stability Score: {stability_score}/100")
print(f"  └─ Verdict: {verdict}")

print(f"\n📋 VARIABILITY ANALYSIS (Coefficient of Variation):")
for metric in ['avg_waiting_time', 'avg_queue_length', 'avg_speed', 'throughput']:
    cv = stability_indicators.get(metric, 0.0)
    stability_label = "🟢 Low" if cv < 0.5 else "🟡 Medium" if cv < 1.0 else "🔴 High"
    print(f"  ├─ {metric}: {cv:.4f} {stability_label}")

print(f"\n📄 OUTPUT FILES:")
print(f"  ├─ JSON Report: {report_path}")
print(f"  ├─ Metrics Plot: {plot_path}")
print(f"  └─ Checkpoint Files: {OUTPUT_DIR}")

print("\n" + "=" * 80)
print(f"VALIDATION COMPLETE")
print("=" * 80)

print(f"\n🎯 FINAL VERDICT: {verdict}")
print(f"   Score: {stability_score}/100")

if verdict == "STABLE":
    print(f"   ✅ System is production-ready.")
    print(f"      - Low exception rate (<{THRESHOLD_EXCEPTION_RATE*100}%)")
    print(f"      - Consistent performance across 1000 steps")
    print(f"      - All subsystems operational")
elif verdict == "PARTIALLY STABLE":
    print(f"   ⚠️  System has minor issues.")
    print(f"      - Some metrics show variability")
    print(f"      - May need tuning for production use")
    for reason in verdict_reasons[:3]:
        print(f"      - {reason}")
else:
    print(f"   ❌ System requires attention.")
    print(f"      - Multiple issues detected")
    print(f"      - Review error logs and metrics")
    for reason in verdict_reasons[:5]:
        print(f"      - {reason}")

print("\n" + "=" * 80)